# Energy Consumption Insights

In [1]:
# Import Libs
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib as jlb 

In [2]:
# Load and explore data
df = pd.read_csv('../data/energy_consumption.csv')

print("Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"\nFirst rows:")
display(df.head())
print(f"\nData Info:")
print(df.info())
print(f"\nBasic Statistics:")
display(df.describe())
print(f"\nMissing Values:")
print(df.isnull().sum())


Dataset Overview:
Shape: (5000, 6)

First rows:


,customer_id,customer_type,regions,building_size_m2,occupants,energy_cost_brl
0,CUSTOMER_0001,residential,Northeast,24,2,64.51
1,CUSTOMER_0002,commercial,Midwest,24,1,55.26
2,CUSTOMER_0003,commercial,Southeast,24,1,74.54
3,CUSTOMER_0004,residential,Northeast,45,4,147.06
4,CUSTOMER_0005,residential,Southeast,45,4,143.06



Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       5000 non-null   object 
 1   customer_type     5000 non-null   object 
 2   regions           5000 non-null   object 
 3   building_size_m2  5000 non-null   int64  
 4   occupants         5000 non-null   int64  
 5   energy_cost_brl   5000 non-null   float64
dtypes: float64(1), int64(2), object(3)
memory usage: 234.5+ KB
None

Basic Statistics:


,building_size_m2,occupants,energy_cost_brl
count,5000.00000,5000.000000,5000.000000
mean,39.57620,2.301800,86.874028
std,17.51638,1.032729,24.383261
min,17.00000,1.000000,52.520000
25%,24.00000,1.000000,68.557500
50%,45.00000,2.000000,83.715000
75%,45.00000,3.000000,98.242500
max,77.00000,4.000000,158.610000



Missing Values:
customer_id         0
customer_type       0
regions             0
building_size_m2    0
occupants           0
energy_cost_brl     0
dtype: int64


## Exploratory Data Analysis

In [3]:
# Energy cost distribution by customer type
fig1 = px.box(
    df,
    x='customer_type',
    y='energy_cost_brl',
    color='customer_type',
    title='Energy Cost Distribution by Customer Type',
    labels={'energy_cost_brl': 'Energy Cost (BRL)', 'customer_type': 'Customer Type'},
    height=500
)
fig1.show()

# Energy cost by region
fig2 = px.bar(
    df.groupby('regions')['energy_cost_brl'].mean().reset_index(),
    x='regions',
    y='energy_cost_brl',
    title='Average Energy Cost by Region',
    labels={'energy_cost_brl': 'Average Energy Cost (BRL)', 'regions': 'Region'},
    color='energy_cost_brl',
    color_continuous_scale='Viridis',
    height=500
)
fig2.show()

# Building size vs energy cost
fig3 = px.scatter(
    df,
    x='building_size_m2',
    y='energy_cost_brl',
    color='customer_type',
    size='occupants',
    title='Energy Cost vs Building Size (bubble size = occupants)',
    labels={'building_size_m2': 'Building Size (m2)', 'energy_cost_brl': 'Energy Cost (BRL)'},
    height=500
)
fig3.show()

# Occupants impact on energy cost
fig4 = px.violin(
    df,
    x='occupants',
    y='energy_cost_brl',
    color='customer_type',
    title='Energy Cost Distribution by Number of Occupants',
    labels={'energy_cost_brl': 'Energy Cost (BRL)', 'occupants': 'Number of Occupants'},
    height=500
)
fig4.show()

print("\nEDA Summary:")
print(f"Average energy cost: BRL {df['energy_cost_brl'].mean():.2f}")
print(f"Residential average: BRL {df[df['customer_type'] == 'residential']['energy_cost_brl'].mean():.2f}")
print(f"Commercial average: BRL {df[df['customer_type'] == 'commercial']['energy_cost_brl'].mean():.2f}")



EDA Summary:
Average energy cost: BRL 86.87
Residential average: BRL 87.15
Commercial average: BRL 86.35


## Predictive Modeling - Decision Tree Regressor

In [4]:
# Prepare data for modeling
df_model = df.copy()

# Encode categorical variables
le_customer = LabelEncoder()
le_region = LabelEncoder()

df_model['customer_type_encoded'] = le_customer.fit_transform(df_model['customer_type'])
df_model['regions_encoded'] = le_region.fit_transform(df_model['regions'])

# Features and target
features = ['customer_type_encoded', 'regions_encoded', 'building_size_m2', 'occupants']
X = df_model[features]
y = df_model['energy_cost_brl']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Train Decision Tree Regressor
dt_model = DecisionTreeRegressor(max_depth=10, min_samples_split=5, random_state=42)
dt_model.fit(X_train, y_train)

# Make predictions
y_pred_train = dt_model.predict(X_train)
y_pred_test = dt_model.predict(X_test)

# Evaluate model
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("\n" + "="*50)
print("MODEL PERFORMANCE")
print("="*50)
print(f"Training R2 Score: {train_r2:.4f}")
print(f"Test R2 Score: {test_r2:.4f}")
print(f"\nTraining MAE: BRL {train_mae:.2f}")
print(f"Test MAE: BRL {test_mae:.2f}")
print(f"\nTraining RMSE: BRL {train_rmse:.2f}")
print(f"Test RMSE: BRL {test_rmse:.2f}")
print("="*50)


Training set size: 4000
Test set size: 1000

MODEL PERFORMANCE
Training R2 Score: 0.6363
Test R2 Score: 0.5985

Training MAE: BRL 12.65
Test MAE: BRL 13.19

Training RMSE: BRL 14.73
Test RMSE: BRL 15.34


In [7]:
# Feature importance visualization
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=True)

# Map encoded names back to original
feature_names_map = {
    'customer_type_encoded': 'Customer Type',
    'regions_encoded': 'Region',
    'building_size_m2': 'Building Size',
    'occupants': 'Occupants'
}
feature_importance['Feature'] = feature_importance['Feature'].map(feature_names_map)

fig_importance = px.bar(
    feature_importance,
    x='Importance',
    y='Feature',
    orientation='h',
    title='Feature Importance in Energy Cost Prediction',
    labels={'Importance': 'Importance Score'},
    color='Importance',
    color_continuous_scale='Reds',
    height=400
)
fig_importance.show()

# Actual vs Predicted for test set
df_results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred_test,
    'Error': np.abs(y_test - y_pred_test)
})

fig_actual_pred = px.scatter(
    df_results,
    x='Actual',
    y='Predicted',
    title='Actual vs Predicted Energy Cost (Test Set)',
    labels={'Actual': 'Actual Cost (BRL)', 'Predicted': 'Predicted Cost (BRL)'},
    color='Error',
    color_continuous_scale='Reds',
    height=500
)
fig_actual_pred.add_shape(type='line', x0=df_results['Actual'].min(), y0=df_results['Actual'].min(),
                          x1=df_results['Actual'].max(), y1=df_results['Actual'].max(),
                          line=dict(dash='dash', color='gray'))
fig_actual_pred.show()

# Residuals plot
df_results['Residuals'] = y_test - y_pred_test
fig_residuals = px.scatter(
    df_results,
    x='Predicted',
    y='Residuals',
    title='Residual Plot (Test Set)',
    labels={'Predicted': 'Predicted Cost (BRL)', 'Residuals': 'Residuals (BRL)'},
    height=500
)
fig_residuals.add_hline(y=0, line_dash='dash', line_color='red')
fig_residuals.show()


## Model Export and Summary

In [6]:
# Export model and encoders
jlb.dump(dt_model, '../models/energy_cost_model.joblib')
jlb.dump(le_customer, '../models/customer_type_encoder.joblib')
jlb.dump(le_region, '../models/region_encoder.joblib')

print("Models exported successfully:")
print("- ../models/energy_cost_model.joblib")
print("- ../models/customer_type_encoder.joblib")
print("- ../models/region_encoder.joblib")

# Create summary report
summary_report = f"""
ENERGY COST PREDICTION MODEL - SUMMARY REPORT

Dataset: 471 customers (Residential & Commercial)
Target Variable: Energy Cost (BRL)

Model: Decision Tree Regressor
- Max Depth: 10
- Min Samples Split: 5

Performance Metrics:
- Test R2 Score: {test_r2:.4f}
- Test MAE: BRL {test_mae:.2f}
- Test RMSE: BRL {test_rmse:.2f}

Top Feature: {feature_importance.iloc[-1]['Feature']} ({feature_importance.iloc[-1]['Importance']:.4f})

Key Insights:
- Occupants and building size are strong cost predictors
- Commercial customers pay average BRL {df[df['customer_type'] == 'commercial']['energy_cost_brl'].mean():.2f}
- Residential customers pay average BRL {df[df['customer_type'] == 'residential']['energy_cost_brl'].mean():.2f}
- Regional variation exists with cost ranging from BRL {df['energy_cost_brl'].min():.2f} to BRL {df['energy_cost_brl'].max():.2f}
"""

print("\n" + summary_report)

# Save summary to file
with open('../summary.txt', 'w') as f:
    f.write(summary_report)

print("Summary saved to ../summary.txt")


Models exported successfully:
- ../models/energy_cost_model.joblib
- ../models/customer_type_encoder.joblib
- ../models/region_encoder.joblib


ENERGY COST PREDICTION MODEL - SUMMARY REPORT

Dataset: 471 customers (Residential & Commercial)
Target Variable: Energy Cost (BRL)

Model: Decision Tree Regressor
- Max Depth: 10
- Min Samples Split: 5

Performance Metrics:
- Test R2 Score: 0.5985
- Test MAE: BRL 13.19
- Test RMSE: BRL 15.34

Top Feature: Occupants (0.9806)

Key Insights:
- Occupants and building size are strong cost predictors
- Commercial customers pay average BRL 86.35
- Residential customers pay average BRL 87.15
- Regional variation exists with cost ranging from BRL 52.52 to BRL 158.61

Summary saved to ../summary.txt
